# CMI Behavior Detection with Multimodal V40 (ws128 only)

This notebook uses a single 5-fold multimodal model (window size 128) for gesture recognition.

In [6]:
import os
from pathlib import Path
import polars as pl

import kaggle_evaluation.cmi_inference_server

from src.inference_pipeline import predict_one


In [7]:
def predict(sequence: pl.DataFrame, demographics: pl.DataFrame) -> str:
    return predict_one(sequence, demographics)


In [ ]:
inference_server = kaggle_evaluation.cmi_inference_server.CMIInferenceServer(predict)

def is_kaggle():
    return "KAGGLE_URL_BASE" in os.environ or Path("/kaggle").exists()

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    if is_kaggle():
        inference_server.run_local_gateway(
            data_paths=(
                '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv',
                '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv',
            )
        )
    else:
        inference_server.run_local_gateway(
            data_paths=(
                '../data/split/test.csv',
                '../data/split/test_demographics.csv',
            )
        )


/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40_128/kaggle_evaluation/core/templates.py:136: RuntimeWarning: 1425 seconds elapsed before server startup.
                This exceeds the startup time limit of 900 seconds that the gateway will enforce
                during the rerun on the hidden test set. Start the server before performing any time consuming steps.
  warnings.warn(
/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40_128/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[flag] = df[cols].isna().all(axis=1)
/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40_128/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 

interp:   0%|          | 0/1 [00:00<?, ?seq/s]

concat:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:utils.pipeline:Building windows (size=128, stride=64, min_len=20)
INFO:utils.pipeline:Windows shapes: X_sensor=(2, 128, 18), X_demo=(2, 7), y=(2,)
INFO:utils.pipeline:Window tensor shape (2, 128, 18)
INFO:utils.pipeline:Building tabular features (wavelet=True, tda=True, tof_event=True, temp_grad=True)
INFO:utils.pipeline:Building ToF windows (size=128, stride=64, min_len=20)
INFO:utils.pipeline:ToF windows shape (2, 128, 5, 8, 8)
INFO:utils.pipeline:Tabular features shape (2, 429)
INFO:utils.pipeline:Building ToF voxel tensor
INFO:utils.pipeline:ToF voxel shape (138, 5, 8, 8)
INFO:utils.pipeline:ToF正規化完了: 有効値23,828個, 欠損値20,332個
INFO:utils.pipeline:Building ToF windows (size=128, stride=64, min_len=20)
INFO:utils.pipeline:ToF windows shape (2, 128, 5, 8, 8)
INFO:utils.pipeline:ToF Windows正規化完了: 有効値23,828個, 欠損値58,092個
INFO:utils.pipeline:Output shapes: windows=(2, 128, 18), demographics=(2, 7), tabular=(2, 429), tof=(138, 5, 8, 8), tof_win=(2, 128, 5, 8, 8)
INFO:utils.pipeline:testデ

✅ TDA処理前: NaN値なし


INFO:utils.pipeline:Transforming dataframe of shape (66, 343)
/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40_128/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[flag] = df[cols].isna().all(axis=1)
/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40_128/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[flag] = df[cols].isna().all(axis=1)
/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40_128/src/utils/feature_engineering.py:171: PerformanceWarning: DataFr

interp:   0%|          | 0/1 [00:00<?, ?seq/s]

In [ ]:
# Check submission file
import pandas as pd
df = pd.read_parquet('submission.parquet')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('First 10 rows:')
print(df.head(10))
print('Value counts:')
print(df['gesture'].value_counts())


Shape: (102, 2)
Columns: ['sequence_id', 'gesture']
First 10 rows:
  sequence_id                    gesture
0  SEQ_025997        Eyelash - pull hair
1  SEQ_002654             Neck - scratch
2  SEQ_049859  Pull air toward your face
3  SEQ_014495          Neck - pinch skin
4  SEQ_004519              Text on phone
5  SEQ_050233              Text on phone
6  SEQ_054258              Text on phone
7  SEQ_051803             Glasses on/off
8  SEQ_061601        Eyebrow - pull hair
9  SEQ_063563         Cheek - pinch skin
Value counts:
gesture
Neck - scratch                                20
Eyelash - pull hair                           13
Text on phone                                 13
Above ear - pull hair                          8
Wave hello                                     8
Feel around in tray and pull out an object     7
Cheek - pinch skin                             5
Forehead - pull hairline                       5
Neck - pinch skin                              4
Write name in air  

In [ ]:
import pandas as pd
from src.utils.cmi_evaluation import calculate_cmi_score

# 例: 推論結果
df_pred = pd.read_parquet('submission.parquet')  # またはcsv等
# 例: 正解ラベル
df_true = pd.read_csv('../data/split/test_labels.csv')

# ラベル名が一致している前提
y_pred = df_pred['gesture'].values
y_true = df_true['gesture'].values

# 必要ならLabelEncoderでエンコード
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(list(y_true) + list(y_pred))
y_true_enc = le.transform(y_true)
y_pred_enc = le.transform(y_pred)

# CMIスコア計算
cmi_score, binary_f1, macro_f1, test_acc = calculate_cmi_score(y_pred_enc, y_true_enc, label_encoder=le, verbose=True)
print(f'CMI Score: {cmi_score:.4f}, Binary F1: {binary_f1:.4f}, Macro F1: {macro_f1:.4f}, Accuracy: {test_acc:.4f}')

CMI評価指標計算開始...
y_true shape: (102,), y_pred shape: (102,)
ラベル変換完了: 102 samples
データ中のジェスチャー: 18種類
Binary分類 - Target: 64, Non-Target: 38
Binary F1: 0.6179
Macro F1: 0.1391
CMI Score: 0.3785
Test Accuracy: 0.0980
CMI Score: 0.3785, Binary F1: 0.6179, Macro F1: 0.1391, Accuracy: 0.0980
